In [ ]:
import event_helpers
import gcsfs
import geopandas as gpd
import google.auth
import pandas as pd
import poi_geography_utils as poi_geog_utils
import world_cup_vars as wc_vars
from gtfs_curator_utils import utils

GCS_FILE_PATH = wc_vars.GCS_FILE_PATH
credentials, _ = google.auth.default()

In [ ]:
STADIUM_NAME = "sofi"
route_gdf = gpd.read_parquet(
    f"{GCS_FILE_PATH}route_summary_{STADIUM_NAME}.parquet",
    storage_options = {"token": credentials.token}
)

In [ ]:
route_gdf.head()

## RT completeness, 
* maybe just show `pct_vp/tu_trips` and `vp/tu_messages_per_minute` for event period?
* not important to show non-event or change

In [ ]:
import gt_extras as gte
from great_tables import GT, md

In [ ]:
def define_rt_columns(rt_type: str):
    return [f"pct_{rt_type}_trips_weekday_event", f"pct_{rt_type}_trips_weekend_event",
      f"{rt_type}_messages_per_minute_weekday_event", f"{rt_type}_messages_per_minute_weekend_event"
    ] 

In [ ]:
route_cols = ["schedule_name", "route_name"]
vp_cols = define_rt_columns("vp")
tu_cols = define_rt_columns("tu")
vp_cols

In [ ]:
rt_table = (
    GT(route_gdf[route_gdf.is_special_route==True][route_cols + vp_cols + tu_cols])
    .cols_label(
        tu_messages_per_minute_weekday_event = "Trip Updates per Minute",
        pct_tu_trips_weekday_event = "% Trips Weekday",
        pct_tu_trips_weekend_event = "% Trips Weekend",
        vp_messages_per_minute_weekday_event = "Vehicle Positions per Minute",
        pct_vp_trips_weekday_event = "% Trips Weekday",
        pct_vp_trips_weekend_event = "% Trips Weekend",
    ).fmt_number(
        columns = ["tu_messages_per_minute_weekday_event", "vp_messages_per_minute_weekday_event"], 
        decimals=1
    ).fmt_percent(
        columns=["pct_tu_trips_weekday_event", "pct_tu_trips_weekend_event", 
                 "pct_vp_trips_weekday_event", "pct_vp_trips_weekend_event"], 
        decimals=1
    ).tab_spanner(
        label="Trip Updates",
        columns=tu_cols
    ).tab_spanner(
        label="Vehicle Positions",
        columns=vp_cols
    )
)

rt_table.pipe(
    gte.gt_color_box, 
    columns=["tu_messages_per_minute_weekday_event", "vp_messages_per_minute_weekday_event"], 
    palette="Blues",
    domain=[0, 4]
).pipe(
    gte.gt_hulk_col_numeric, 
    columns=["pct_tu_trips_weekday_event", "pct_tu_trips_weekend_event", 
             "pct_vp_trips_weekday_event", "pct_vp_trips_weekend_event"],
    palette=["#FFEC8B", "#E5F5E0"], #[light goldenrod1, white alyssum (from greens)]
    domain=[0, 1],
    alpha=0.1
)

In [ ]:
trips_wide_gdf_near[trips_wide_gdf_near.is_special_route==False].combined_change_pct_vp_trips.hist()

In [ ]:
trips_wide_gdf_near[trips_wide_gdf_near.is_special_route==True].combined_change_pct_vp_trips.hist()